# SiO2 extraction walkthrough (sio2_dish_white_20 + bare_silicon)

Package-based scratchpad for the extraction front-end:
load → extract silicon pieces → isolate the SiO2 film → compare preprocessing.

For *interactive* tuning use `debug_masks.py` (pieces), `debug_film.py` (SiO2) and
`debug_preprocess.py` (filters) — see `docs/debug_tools.md`. This notebook shows the
same steps via the `hsi_workflow` API. The raw cubes are large; we work on a crop.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace

from hsi_workflow.config import DATASETS, PieceConfig, FilmConfig, PreprocessConfig
from hsi_workflow.cube_io import Cube, load_cube, iter_cube_paths, load_reference_spectrum
from hsi_workflow.preprocessing import calibrate_reflectance, savgol_smooth, normalize_intensity, noise_metrics
from hsi_workflow.pieces import extract_pieces
from hsi_workflow.film import extract_film, film_distance

TARGET = 'sio2_dish_white_20'   # the SiO2 dish scan
CONTROL = 'sio2_bare_si'        # bare-silicon reference population
CROP = (0, 700, 0, 700)         # work on a window for speed; set None for the full scan
%matplotlib inline

In [ ]:
def load_reflectance(dataset_name, crop=None):
    ds = DATASETS[dataset_name]
    label, hdr = iter_cube_paths(ds)[0]
    cube = load_cube(hdr)
    data = cube.data
    if crop:
        r0, r1, c0, c1 = crop
        data = data[r0:r1, c0:c1, :]
    white, sw = load_reference_spectrum(ds.white_ref)
    dark, sd = load_reference_spectrum(ds.dark_ref)
    refl = calibrate_reflectance(data, cube.shutter, white, sw, dark, sd)
    return Cube(data=refl, wavelengths=cube.wavelengths, shutter=1.0, ceiling=cube.ceiling,
                path=hdr, label=label, material=ds.material)

target = load_reflectance(TARGET, CROP)
print('target cube:', target.data.shape, 'wavelengths', target.wavelengths[[0, -1]])

## 1. Extract silicon pieces

On the white dish the auto border-frame background is contaminated. If the mask is
messy, set `background_bbox` to a clean empty-dish region (find it visually in
`debug_masks.py` with the `R` box), and/or enable `flat_field` / `on_reflectance`.

In [ ]:
cfg = PieceConfig(method='sam', threshold='otsu')
# Example quality overrides (tune in debug_masks.py):
# cfg = replace(cfg, background_bbox=(0, 40, 0, 40), flat_field=True)
pieces = extract_pieces(target, cfg)
print(f'{len(pieces)} pieces')

p = pieces[0]
band = p.data[:, :, p.n_bands // 2]
plt.figure(figsize=(5, 5))
plt.imshow(band, cmap='gray')
plt.imshow(np.where(p.mask, 1.0, np.nan), cmap='autumn', alpha=0.35, vmin=0, vmax=1)
plt.title(f'{p.piece_id}: mask overlay'); plt.axis('off'); plt.show()

## 2. Isolate the SiO2 film within a piece

Reference against **bare silicon**: `control` uses the mean spectrum of the
`sio2_bare_si` dataset, `in_piece` uses the wafer's own 2-cluster split.

In [ ]:
control = load_reflectance(CONTROL, None)
cpieces = extract_pieces(control, PieceConfig())
bare_ref = np.vstack([cp.data[cp.mask] for cp in cpieces]).mean(axis=0)
print('bare-Si reference spectrum:', bare_ref.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
band = p.data[:, :, p.n_bands // 2]
for ax, ref_mode in zip(axes[:2], ['control', 'in_piece']):
    fm = extract_film(p, FilmConfig(reference=ref_mode, min_area=30, open_iter=1, close_iter=1), ref_spectrum=bare_ref)
    ax.imshow(band, cmap='gray')
    ax.imshow(np.where(fm.sio2_mask, 1.0, np.nan), cmap='autumn', alpha=0.45, vmin=0, vmax=1)
    frac = fm.sio2_mask.sum() / max(1, p.mask.sum())
    ax.set_title(f'SiO2 ({ref_mode})  {frac:.0%} of wafer'); ax.axis('off')
dist = film_distance(p.data, p.mask, bare_ref, FilmConfig(reference='control'))
axes[2].imshow(np.where(p.mask, dist, np.nan), cmap='magma'); axes[2].set_title('film distance'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 3. Compare preprocessing on a pixel

Savitzky-Golay smoothing + SNV, with before/after noise metrics.

In [ ]:
wl = np.asarray(target.wavelengths, float)
rr, cc = np.argwhere(p.mask)[len(np.argwhere(p.mask)) // 2]
refl = p.data[rr, cc, :]
sm = savgol_smooth(refl[None, None, :], 11, 2)[0, 0, :]
snv = normalize_intensity(sm[None, None, :], 'snv')[0, 0, :]

nb = noise_metrics(p.data, 11, 2, sample=2000)
na = noise_metrics(savgol_smooth(p.data, 11, 2), 11, 2, sample=2000)
print(f"RMS noise  before {nb['rms_noise']:.4g}  after {na['rms_noise']:.4g}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(wl, refl, '0.6', label='reflectance'); ax[0].plot(wl, sm, 'r', label='SG smoothed')
ax[0].set_title('smoothing'); ax[0].legend(); ax[0].set_xlabel('nm')
ax[1].plot(wl, snv, 'tab:blue'); ax[1].set_title('SG + SNV'); ax[1].set_xlabel('nm')
plt.tight_layout(); plt.show()

## Next steps

- Tune the knobs visually (`debug_masks.py`, `debug_film.py`, `debug_preprocess.py`),
  press `p`, and paste the printed configs into `hsi_workflow/config.py`.
- Build the organized dataset: `run_organize --datasets sio2_bare_si sio2_dish_white_20`.
- Analyze oxide-only: `run_analyze --target sio2_dish_white_20 --baseline sio2_bare_si --extract-film`.